In [16]:
import random
from datetime import datetime

#3
class Item:
    def __init__(self, item_id, name, power):
        self.id = item_id
        self.name = name.strip().title()
        self.power = power

    #3
    def __eq__(self, other):
        return isinstance(other, Item) and self.id == other.id

    def __hash__(self):
        return hash(self.id)

    def __str__(self):
        return f"Item(id={self.id}, name='{self.name}', power={self.power})"

#4,
class Inventory:
    def __init__(self):
        self._items = {}

    def add_item(self, item: Item):
        self._items[item.id] = item

    def remove_item(self, item_id: int):
        if item_id in self._items:
            del self._items[item_id]

    def get_items(self) -> list[Item]:
        return list(self._items.values())

    def to_dict(self) -> dict[int, Item]:
        return self._items

    #5
    def get_strong_items(self, min_power: int):
        check_power = lambda item: item.power >= min_power
        return [item for item in self._items.values() if check_power(item)]

    #18
    def __iter__(self):
        return iter(self._items.values())

#6
class Event:
    def __init__(self, event_type: str, data: dict):
        self.type = event_type
        self.data = data
        self.timestamp = datetime.now()

    def __str__(self):
        ts = self.timestamp.strftime('%Y-%m-%d %H:%M:%S')
        return f"Event(type='{self.type}', data={self.data}, timestamp='{ts}')"

#1
class Player:
    def __init__(self, id, name, hp):
        self._id = id
        self._name = name.strip().title()
        self._hp = max(hp, 0)
        self._inventory = Inventory()

    #16
    @property
    def hp(self):
        return self._hp

    @hp.setter
    def hp(self, value):
        self._hp = max(0, value)

    #7
    def handle_event(self, event: Event):
        if event.type == "ATTACK":
            damage = event.data.get("damage", 0)
            self.hp -= damage
        elif event.type == "HEAL":
            amount = event.data.get("amount", 0)
            self.hp += amount
        elif event.type == "LOOT":
            item = event.data.get("item")
            if item:
                self._inventory.add_item(item)

    #2
    @classmethod
    def from_string(cls, data):
        parts = [part.strip() for part in data.split(',')]
        return cls(int(parts[0]), parts[1], int(parts[2]))

    def __str__(self): # Задача 1
        return f"Player(id={self._id}, name='{self._name}', hp={self._hp})"

    def __del__(self): # Задача 1 & 17: Деструктор
        print(f"Player {self._name} удалён")

#15
class Warrior(Player):
    def handle_event(self, event: Event):
        if event.type == "ATTACK":
            # Уменьшение урона на 10%
            event.data["damage"] = int(event.data.get("damage", 0) * 0.9)
        super().handle_event(event)

class Mage(Player):
    def handle_event(self, event: Event):
        if event.type == "LOOT" and "item" in event.data:
            # Усиление предмета на 10%
            event.data["item"].power = int(event.data["item"].power * 1.1)
        super().handle_event(event)

#8
class Logger:
    @staticmethod
    def log(event: Event, player_id: int, filename: str):
        with open(filename, "a", encoding="utf-8") as f:
            f.write(f"{event.timestamp};{player_id};{event.type};{event.data}\n")
#9
    @staticmethod
    def read_logs(filename: str) -> list[Event]:
        events = []
        try:
            with open(filename, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split(';')
                    events.append(Event(parts[2], eval(parts[3])))
        except FileNotFoundError: pass
        return events

#10
class EventIterator:
    def __init__(self, events: list[Event]):
        self._events = events
        self._index = 0

    def __iter__(self): return self

    def __next__(self):
        if self._index < len(self._events):
            res = self._events[self._index]
            self._index += 1
            return res
        raise StopIteration

#11
def damage_stream(events: list[Event]):
    for event in events:
        if event.type == "ATTACK":
            yield event.data.get("damage", 0)

#12
def generate_events(players, items, n):
    events = []
    #14
    decide_action = lambda hp: "HEAL" if hp < 30 else random.choice(["ATTACK", "LOOT"])

    for p in players:
        for _ in range(n):
            etype = decide_action(p.hp)
            if etype == "ATTACK": data = {"damage": random.randint(10, 40)}
            elif etype == "HEAL": data = {"amount": random.randint(10, 20)}
            else: data = {"item": random.choice(items)}
            events.append(Event(etype, data))
    return events

#13
def analyze_logs(events: list[Event]):
    dmg_list = list(damage_stream(events))
    types = [e.type for e in events]
    return {
        "total_damage": sum(dmg_list),
        "most_common_event": max(set(types), key=types.count) if types else None
    }

#19
def analyze_inventory(inventories: list[Inventory]):
    all_items = []
    for inv in inventories:
        all_items.extend(inv.get_items())
    return {
        "unique_items": set(all_items),
        "top_power": max(all_items, key=lambda x: x.power) if all_items else None
    }

#20
def main():
    warrior = Warrior(1, " Gimli ", 100)
    mage = Mage(2, " Gandalf ", 80)
    items = [Item(10, "Axe", 50), Item(11, "Staff", 40)]
    events = generate_events([warrior, mage], items, 5)
    for e in events:
        warrior.handle_event(e)
        mage.handle_event(e)
        Logger.log(e, warrior._id, "game_logs.txt")

    print(f"--- Результаты симуляции ---")
    print(warrior)
    print(mage)
    print(f"Аналитика событий: {analyze_logs(events)}")
    print(f"Топ предмет: {analyze_inventory([warrior._inventory, mage._inventory])['top_power']}")

if __name__ == "__main__":
    main()

--- Результаты симуляции ---
Player(id=1, name='Gimli', hp=0)
Player(id=2, name='Gandalf', hp=0)
Аналитика событий: {'total_damage': 112, 'most_common_event': 'LOOT'}
Топ предмет: Item(id=10, name='Axe', power=60)
Player Gandalf удалён
Player Gimli удалён
